[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raikar8/ML-Lab/blob/main/notebooks/04_torch_compile_profiling.ipynb)

# 4 · `torch.compile`, graph breaks and the profiler
Week 9. Measure eager vs compiled on a small decoder, read a profiler table, and see what a graph break costs.

In [ ]:
# Colab: Runtime → Change runtime type → GPU (a T4 is enough except where noted). Kaggle: Settings → Accelerator.
import os, sys
if not os.path.exists("ML-Lab") and not os.path.exists("gpu"):
    !git clone --depth 1 https://github.com/raikar8/ML-Lab.git
if os.path.exists("ML-Lab"):
    %cd ML-Lab
sys.path.insert(0, ".")
import torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Switch the runtime to a GPU"

In [ ]:
from gpu.decode import TinyDecoder
m = TinyDecoder(vocab=32000, d=1024, n_layers=8, n_heads=16, max_len=1024).cuda().half().eval()
x = torch.randint(0, 32000, (8, 512), device="cuda")
def timed(f, n=20):
    for _ in range(3): f()
    torch.cuda.synchronize(); s, e = torch.cuda.Event(True), torch.cuda.Event(True); s.record()
    for _ in range(n): f()
    e.record(); torch.cuda.synchronize(); return s.elapsed_time(e) / n
with torch.inference_mode():
    eager = timed(lambda: m.forward_full(x))
    cm = torch.compile(m.forward_full)
    compiled = timed(lambda: cm(x))
print(f"eager {eager:.2f} ms   compiled {compiled:.2f} ms   speed-up {eager / compiled:.2f}x")

In [ ]:
from torch.profiler import profile, ProfilerActivity
with torch.inference_mode(), profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    m.forward_full(x); torch.cuda.synchronize()
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=12))

## Graph breaks
`.item()` forces a GPU→CPU sync and Dynamo can't trace data-dependent Python control flow, so it splits the graph.

In [ ]:
import torch._dynamo as dynamo
def f(x):
    y = torch.relu(x) * 2
    if y.sum().item() > 0:      # graph break: data-dependent Python branch
        y = y + 1
    return torch.tanh(y)
explanation = dynamo.explain(f)(torch.randn(1000, device="cuda"))
print("graphs:", explanation.graph_count, " breaks:", explanation.graph_break_count)
for b in explanation.break_reasons: print(" -", b.reason[:120])
g = lambda x: torch.tanh(torch.where((torch.relu(x) * 2).sum() > 0, torch.relu(x) * 2 + 1, torch.relu(x) * 2))
print("rewritten:", dynamo.explain(g)(torch.randn(1000, device="cuda")).graph_break_count, "breaks")

## Your turn
Export a Perfetto trace (`prof.export_chrome_trace("trace.json")`), open it at ui.perfetto.dev, and find: the largest kernel, any
gaps where the GPU idles, and how many kernels a single decoder layer launches eager vs compiled.